In [ ]:
# Notebook 02 — Preparación y Limpieza del Corpus

**Reto 7 — Análisis de Datos No Estructurados y Redes Sociales**

## 🎯 Objetivo
Limpiar, tokenizar, eliminar stopwords y lematizar el corpus de tweets.

## 📌 Input
- `data/processed/tweets_ia_raw_20260920.csv` (del Notebook 01)

## 📌 Output
- `data/processed/tweets_limpios_YYYYMMDD.csv` — Texto limpio
- `data/processed/tweets_nlp_YYYYMMDD.csv` — Corpus final para NLP

In [8]:
# === Imports ===
import re
import sys
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

# NLP
import nltk
import spacy
from nltk.corpus import stopwords

# Activar tqdm para pandas
tqdm.pandas()

# === Configuración ===
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 150)
sns.set_style("whitegrid")

# ============================================
# BUSCAR LA RAÍZ DEL PROYECTO AUTOMÁTICAMENTE
# ============================================
def encontrar_raiz_proyecto():
    cwd = Path.cwd()
    for candidato in [cwd, cwd.parent, cwd.parent.parent, cwd.parent.parent.parent]:
        if (candidato / "requirements.txt").exists() and (candidato / "data").exists():
            return candidato
    return cwd.parent

BASE_DIR = encontrar_raiz_proyecto()
DATA_RAW = BASE_DIR / "data" / "raw" / "tweets"
DATA_PROCESSED = BASE_DIR / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"📍 cwd: {Path.cwd()}")
print(f"📁 BASE_DIR: {BASE_DIR}")
print(f"📁 DATA_PROCESSED: {DATA_PROCESSED}")
print(f"\n📂 Archivos en processed:")
for f in DATA_PROCESSED.glob("*"):
    print(f"   - {f.name} ({f.stat().st_size / 1024 / 1024:.2f} MB)")

📍 cwd: C:\Users\jdthg\Documents\curso6BI\reto7_Analisis_datos_no_estructurados
📁 BASE_DIR: C:\Users\jdthg\Documents\curso6BI\reto7_Analisis_datos_no_estructurados
📁 DATA_PROCESSED: C:\Users\jdthg\Documents\curso6BI\reto7_Analisis_datos_no_estructurados\data\processed

📂 Archivos en processed:
   - .gitkeep (0.00 MB)
   - tweets_ia_raw_20260920.csv (116.66 MB)


In [9]:
# ============================================
# Cargar el corpus procesado del Notebook 01
# ============================================

archivos = list(DATA_PROCESSED.glob("tweets_ia_raw_*.csv")) + \
           list(DATA_PROCESSED.glob("tweets_ia_raw_*.parquet"))

if not archivos:
    print("⚠️ No hay archivo procesado. Cargando directo del raw...")
    RUTA = DATA_RAW / "Twitter Jan Mar.csv"
    df = pd.read_csv(RUTA, encoding="utf-8")
else:
    archivos.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    RUTA = archivos[0]
    print(f"📂 Cargando: {RUTA.name}")
    if RUTA.suffix == ".parquet":
        df = pd.read_parquet(RUTA)
    else:
        df = pd.read_csv(RUTA, encoding="utf-8-sig")

print(f"✅ Shape: {df.shape}")
print(f"📋 Columnas: {list(df.columns)}")
display(df.head(3))

📂 Cargando: tweets_ia_raw_20260920.csv
✅ Shape: (500036, 7)
📋 Columnas: ['date', 'id', 'content', 'username', 'like_count', 'retweet_count', '_len_content']


,date,id,content,username,like_count,retweet_count,_len_content
0,2023-03-29 22:58:21+00:00,1641213230730051584,"Free AI marketing and automation tools, strategies, and collaboration launching new week https://t.co/Qwti8LfBpb #ChatGPT",RealProfitPros,0.0,0.0,123.0
1,2023-03-29 22:58:18+00:00,1641213218520481805,@MecoleHardman4 Chat GPT says it’s 15. 😂,AmyLouWho321,0.0,0.0,40.0
2,2023-03-29 22:57:53+00:00,1641213115684536323,"https://t.co/FjJSprt0te - Chat with any PDF!\nCheck out how this new AI quickly answers questions from your PDFs.\nPerfect for students, researche...",yjleon1976,0.0,0.0,201.0


In [10]:
# ============================================
# Configurar recursos NLP
# ============================================

# Stopwords combinadas
try:
    stopwords_es = set(stopwords.words("spanish"))
    stopwords_en = set(stopwords.words("english"))
except LookupError:
    nltk.download("stopwords")
    stopwords_es = set(stopwords.words("spanish"))
    stopwords_en = set(stopwords.words("english"))

# Stopwords custom del dominio
stopwords_custom = {
    "rt", "via", "amp", "get", "got", "go", "make", "made", "use", "used",
    "using", "want", "need", "know", "think", "see", "like", "say", "said",
    "one", "two", "new", "good", "great", "best", "today", "now", "still",
    "ser", "estar", "tener", "hacer", "decir", "poder", "querer", "saber",
    "ver", "dar", "ir", "venir", "llegar", "pasar", "quedar",
    "chatgpt", "openai", "gpt", "https", "http", "www", "com", "t", "co",
}

stopwords_total = stopwords_es | stopwords_en | stopwords_custom
print(f"📚 Stopwords español: {len(stopwords_es):,}")
print(f"📚 Stopwords inglés:  {len(stopwords_en):,}")
print(f"📚 Stopwords custom:  {len(stopwords_custom):,}")
print(f"📚 TOTAL combinadas:  {len(stopwords_total):,}")

# SpaCy español
try:
    nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
    print("✅ SpaCy es_core_news_sm cargado")
except OSError:
    print("⚠️ Instalando modelo...")
    import subprocess
    subprocess.run([sys.executable, "-m", "spacy", "download", "es_core_news_sm"])
    nlp = spacy.load("es_core_news_sm", disable=["parser", "ner"])
    print("✅ SpaCy listo")

📚 Stopwords español: 313
📚 Stopwords inglés:  198
📚 Stopwords custom:  52
📚 TOTAL combinadas:  553
✅ SpaCy es_core_news_sm cargado


In [11]:
# ============================================
# Funciones de limpieza
# ============================================

def quitar_urls(texto):
    return re.sub(r"http\S+|www\.\S+|t\.co/\S+", "", texto)

def quitar_menciones(texto):
    return re.sub(r"@\w+", "", texto)

def quitar_hashtags(texto):
    return re.sub(r"#(\w+)", r"\1", texto)

def quitar_numeros(texto):
    return re.sub(r"\b\d+\b", "", texto)

def quitar_emojis(texto):
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F700-\U0001F77F"
        "\U0001F780-\U0001F7FF"
        "\U0001F800-\U0001F8FF"
        "\U0001F900-\U0001F9FF"
        "\U0001FA00-\U0001FA6F"
        "\U0001FA70-\U0001FAFF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE
    )
    return emoji_pattern.sub("", texto)

def limpiar_texto(texto):
    """Pipeline completo de limpieza."""
    if not isinstance(texto, str):
        return ""
    texto = texto.lower()
    texto = quitar_urls(texto)
    texto = quitar_menciones(texto)
    texto = quitar_hashtags(texto)
    texto = quitar_emojis(texto)
    texto = quitar_numeros(texto)
    texto = re.sub(r"[^a-záéíóúñü\s]", " ", texto)
    texto = re.sub(r"\s+", " ", texto).strip()
    return texto

# Prueba
ejemplo = "¡Hola @maria! Mira esto #ChatGPT https://t.co/abc123 😀 Es INCREÍBLE!!! 2024"
print(f"Original: {ejemplo}")
print(f"Limpio:   {limpiar_texto(ejemplo)}")

Original: ¡Hola @maria! Mira esto #ChatGPT https://t.co/abc123 😀 Es INCREÍBLE!!! 2024
Limpio:   hola mira esto chatgpt es increíble


In [12]:
# ============================================
# Aplicar limpieza a todo el corpus
# ============================================
print("🧹 Limpiando texto...")
df["texto_limpio"] = df["content"].progress_apply(limpiar_texto)

# Verificar
print(f"\n📊 Longitud media antes: {df['content'].str.len().mean():.1f}")
print(f"📊 Longitud media después: {df['texto_limpio'].str.len().mean():.1f}")

print(f"\n👀 Ejemplos:")
for i in range(3):
    print(f"\n🔹 Original: {df['content'].iloc[i][:120]}")
    print(f"   Limpio:   {df['texto_limpio'].iloc[i][:120]}")

🧹 Limpiando texto...


  0%|          | 0/500036 [00:00<?, ?it/s]


📊 Longitud media antes: 167.3
📊 Longitud media después: 134.0

👀 Ejemplos:

🔹 Original: Free AI marketing and automation tools, strategies, and collaboration launching new week https://t.co/Qwti8LfBpb   #Chat
   Limpio:   free ai marketing and automation tools strategies and collaboration launching new week chatgpt

🔹 Original: @MecoleHardman4 Chat GPT says it’s 15. 😂
   Limpio:   chat gpt says it s

🔹 Original: https://t.co/FjJSprt0te - Chat with any PDF!
Check out how this new AI quickly answers questions from your PDFs.
Perfect
   Limpio:   chat with any pdf check out how this new ai quickly answers questions from your pdfs perfect for students researchers an


In [13]:
# ============================================
# Tokenizar y filtrar stopwords
# ============================================
print("🔤 Tokenizando...")

def tokenizar_y_filtrar(texto):
    if not texto:
        return []
    tokens = texto.split()
    return [t for t in tokens if t not in stopwords_total and len(t) >= 3]

# Muestra manejable (ajusta según RAM)
LIMITE = 100_000
if LIMITE and len(df) > LIMITE:
    print(f"⚠️ Muestra de {LIMITE:,} tweets (de {len(df):,})")
    df_work = df.head(LIMITE).copy()
else:
    df_work = df.copy()

df_work["tokens"] = df_work["texto_limpio"].progress_apply(tokenizar_y_filtrar)

print(f"\n📊 Tokens totales: {df_work['tokens'].str.len().sum():,}")
print(f"📊 Media tokens/tweet: {df_work['tokens'].str.len().mean():.1f}")

# Vocabulario
vocabulario = set()
for tokens in df_work["tokens"]:
    vocabulario.update(tokens)
print(f"📚 Vocabulario único: {len(vocabulario):,} palabras")

print(f"\n👀 Ejemplos de tokens:")
for i in range(3):
    print(f"   {df_work['tokens'].iloc[i][:10]}")

🔤 Tokenizando...
⚠️ Muestra de 100,000 tweets (de 500,036)


  0%|          | 0/100000 [00:00<?, ?it/s]


📊 Tokens totales: 1,154,110
📊 Media tokens/tweet: 11.5
📚 Vocabulario único: 62,936 palabras

👀 Ejemplos de tokens:
   ['free', 'marketing', 'automation', 'tools', 'strategies', 'collaboration', 'launching', 'week']
   ['chat', 'says']
   ['chat', 'pdf', 'check', 'quickly', 'answers', 'questions', 'pdfs', 'perfect', 'students', 'researchers']


In [14]:
# ============================================
# Lematización con SpaCy
# ============================================
print("🌿 Lematizando con SpaCy... (puede tardar unos minutos)")

# Convertir tokens a strings
textos_para_lematizar = df_work["tokens"].apply(lambda x: " ".join(x) if x else "").tolist()

# Procesar en lotes
lemas_resultado = []
for doc in tqdm(nlp.pipe(textos_para_lematizar, batch_size=500), total=len(textos_para_lematizar)):
    lemas = [
        token.lemma_.lower() for token in doc
        if not token.is_stop and not token.is_punct and len(token.lemma_) >= 3
    ]
    lemas_resultado.append(lemas)

df_work["lemas"] = lemas_resultado

print(f"\n✅ Lematización completa")
print(f"📊 Media lemas/tweet: {df_work['lemas'].str.len().mean():.1f}")
print(f"📊 Total lemas: {df_work['lemas'].str.len().sum():,}")

# Vocabulario lematizado
vocab_lemas = set()
for lemas in df_work["lemas"]:
    vocab_lemas.update(lemas)
print(f"📚 Vocabulario lematizado: {len(vocab_lemas):,}")

🌿 Lematizando con SpaCy... (puede tardar unos minutos)


  0%|          | 0/100000 [00:00<?, ?it/s]


✅ Lematización completa
📊 Media lemas/tweet: 11.5
📊 Total lemas: 1,151,749
📚 Vocabulario lematizado: 65,367


In [16]:
# ============================================
# Guardar corpus procesado (versión CSV)
# ============================================
timestamp = datetime.now().strftime("%Y%m%d")

# Guardar texto limpio (con todos los campos)
ruta_limpio = DATA_PROCESSED / f"tweets_limpios_{timestamp}.csv"
df_work.to_csv(ruta_limpio, index=False, encoding="utf-8-sig")
print(f"✅ Guardado limpio: {ruta_limpio}")

# Guardar solo columnas clave para NLP
cols_nlp = ["date", "username", "texto_limpio", "tokens", "lemas",
            "like_count", "retweet_count"]
cols_nlp = [c for c in cols_nlp if c in df_work.columns]

ruta_nlp = DATA_PROCESSED / f"tweets_nlp_{timestamp}.csv"
df_work[cols_nlp].to_csv(ruta_nlp, index=False, encoding="utf-8-sig")
print(f"✅ Guardado NLP: {ruta_nlp}")

print(f"\n📊 Shape final: {df_work.shape}")
print(f"\n📂 Contenido de processed:")
for f in DATA_PROCESSED.glob("tweets_*"):
    print(f"   - {f.name} ({f.stat().st_size/1024/1024:.2f} MB)")

✅ Guardado limpio: C:\Users\jdthg\Documents\curso6BI\reto7_Analisis_datos_no_estructurados\data\processed\tweets_limpios_20260920.csv
✅ Guardado NLP: C:\Users\jdthg\Documents\curso6BI\reto7_Analisis_datos_no_estructurados\data\processed\tweets_nlp_20260920.csv

📊 Shape final: (100000, 10)

📂 Contenido de processed:
   - tweets_ia_raw_20260920.csv (116.66 MB)
   - tweets_limpios_20260920.csv (60.70 MB)
   - tweets_nlp_20260920.csv (41.49 MB)


In [ ]:
## ✅ Conclusión

- **Tweets procesados:** [N] (muestra de 100k si el corpus era mayor)
- **Pipeline aplicado:** limpieza → tokenización → stopwords → lematización
- **Vocabulario final:** ~[N] lemas únicos
- **Archivos generados:**
  - `tweets_limpios_YYYYMMDD.csv`
  - `tweets_nlp_YYYYMMDD.parquet`

## 🎯 Siguiente paso
**Notebook 03 — EDA y visualización:**
- Nube de palabras
- Frecuencia de términos
- Análisis temporal
- Hashtags más usados